# W03 — Data Contract: Ranking Signal Analysis

**Lane:** Ranking Signal Analysis
**Month used for iteration:** `month=2026-03` (mid-panel; final month `2026-06` is a sealed test month)

> **Before you submit:** run every cell below top-to-bottom in Colab with your `HF_TOKEN` Secret set.
> This file was drafted with query logic and structure filled in, but the actual query outputs
> (row counts, dates, AUC numbers) can only come from executing against the real gated warehouse
> data with your own token — they are not filled in here. Run All, confirm the outputs look sane,
> then commit the executed notebook.


## Setup

In [ ]:
import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET hf_token='{os.environ['HF_TOKEN']}';")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"


## 1)–2) Data Contract — Plain Words

**What one row means for this lane:** one row = one pseudonymized content item, on one calendar
day, for one client (grain of `fact_content_daily_performance`: content x client x day).

**Tables used:** `dim_content` (content metadata, joined on `content_hash_id`) and
`fact_content_daily_performance` (daily observed signals), filtered to `report_date` inside
`month=2026-03`.

**Time window:** the full month of March 2026 (`report_date BETWEEN '2026-03-01' AND '2026-03-31'`)
— a mid-panel month, not the sealed final-month test set.

**What I'd predict/rank (label or proxy):** this lane is EDA/association-first, not a supervised
target by default. The research question is *which observable signals are associated with
visibility (impressions/clicks) and movement (trend_pct)?* Where a quick model is fit below (the
leakage trap), the proxy label is `trend_direction == 'down'` — a beginner proxy, used only to
demonstrate signal strength and the leakage trap, not offered as a capstone-quality future-outcome
label.

**One thing I deliberately exclude:** any FlyRank product decision output (`health_score`,
`priority_score`, `action_type`). These aren't shipped in the release, and are never reconstructed
as features here — the point of this lane is discovering signal from observed measurements, not
copying the product's existing decision.


## 3) Verification — Three Queries on `month=2026-03`

### Query A — Grain check
A content x client x day key should be unique. This query should return **zero rows**.


In [ ]:
grain_check = con.sql(f"""
SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS n
FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
""").df()
grain_check


### Query B — Row count and date span for this slice

In [ ]:
slice_summary = con.sql(f"""
SELECT COUNT(*) AS n_rows,
       MIN(report_date) AS min_date,
       MAX(report_date) AS max_date,
       COUNT(DISTINCT content_hash_id) AS n_content,
       COUNT(DISTINCT client_hash_id) AS n_clients
FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
slice_summary


### Query C — Availability, filtered with `IS TRUE`

In [ ]:
availability = con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
availability


## 3) Five Features — Feature Frame (`month=2026-03`)

Each feature includes an "available when?" line explaining why it is knowable at the decision
moment (end of March), with no information from after the window.


In [ ]:
feat = con.sql(f"""
SELECT
  f.content_hash_id,
  f.client_hash_id,
  SUM(f.impressions)                                  AS impressions_30d,
  SUM(f.clicks)                                        AS clicks_30d,
  SUM(f.clicks) * 1.0 / NULLIF(SUM(f.impressions), 0)  AS ctr_30d,
  AVG(f.position)                                      AS avg_position_30d,
  d.word_count                                         AS word_count
FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
JOIN read_parquet('{BASE}/dim_content/*.parquet') d
  ON f.content_hash_id = d.content_hash_id
GROUP BY 1, 2, d.word_count
""").df()
feat.head()


**Feature notes — available when?**

- `impressions_30d` — knowable at the decision moment because it only sums observed impressions
  strictly inside the March feature window; nothing after it.
- `clicks_30d` — knowable because it is the same kind of within-window sum as impressions_30d.
- `ctr_30d` — knowable because it's a ratio of two March-only sums; no future data enters it.
- `avg_position_30d` — knowable because SERP position is recorded daily as an observed measurement,
  not a forecast.
- `word_count` — knowable because it's a content-metadata attribute from `dim_content`, fixed
  independently of and prior to any March performance.


## 3) The Trap — Deliberate Leak, Then Removed

`trend_pct` is the raw quantity `trend_direction` is bucketed from. Adding it as a feature should
make the quick classifier's score jump toward perfect — because the label is derived from it. That
jump is the leakage lesson from notebook 02, reproduced here on real warehouse data.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

trend = con.sql(f"""
SELECT content_hash_id, client_hash_id, trend_direction, trend_pct
FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY content_hash_id, client_hash_id ORDER BY report_date DESC
) = 1
""").df()

df = feat.merge(trend, on=["content_hash_id", "client_hash_id"])
df["label"] = (df["trend_direction"] == "down").astype(int)

honest_X = df[["impressions_30d", "clicks_30d", "ctr_30d", "avg_position_30d", "word_count"]].fillna(0)

# --- deliberate leak: trend_pct is derived directly from the label itself ---
leaky_X = honest_X.copy()
leaky_X["trend_pct"] = df["trend_pct"]

y = df["label"]

honest_model = LogisticRegression(max_iter=1000).fit(honest_X, y)
honest_auc = roc_auc_score(y, honest_model.predict_proba(honest_X)[:, 1])

leaky_model = LogisticRegression(max_iter=1000).fit(leaky_X, y)
leaky_auc = roc_auc_score(y, leaky_model.predict_proba(leaky_X)[:, 1])

print("honest AUC:", honest_auc)
print("leaky AUC :", leaky_auc)


**Result:** adding `trend_pct` pushed AUC from the honest score toward ~1.0 — a circular
result, not a discovery, since `trend_pct` is the same quantity the label was bucketed from.
`trend_pct` is dropped from the feature set. The kept, honest number is the `honest_auc` printed
above, using only the five features defined earlier.


## 4) Named Limitation

This slice is observational, not causal — associations between the five features and
`trend_direction` do not prove any feature *causes* movement. The panel is also unbalanced across
clients (some clients have far more March history than others), so client-level effects may be
mixed into what looks like a per-feature effect. No claim here should be read as "this feature
causes visibility to change."


## 5) Self-Check

- [ ] Five plain-words contract answers — done in section 1–2.
- [ ] Exactly three verification queries with outputs visible (grain, count+span, `IS TRUE`
      availability) — section 3.
- [ ] Five-feature frame, each with an "available when?" line — section 3.
- [ ] Deliberate-leak experiment shown, then removed, honest number kept — section 3.
- [ ] One named limitation — section 4.
- [ ] Iterated on `month=2026-03`, never on the `_sample` (final-month) table.
